In [1]:
import pandas as pd
import duckdb

In [2]:
con = duckdb.connect()

con.execute("""
CREATE OR REPLACE TABLE paysim AS
SELECT *
FROM read_csv_auto('../data/paysim_transactions.csv');
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
con.execute("""
SELECT COUNT(*) AS total_rows
FROM paysim;
""").df()

,total_rows
0,6362620


In [4]:
type_risk = con.execute("""
SELECT
    type,
    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_transactions,
    ROUND(SUM(isFraud) * 1.0 / COUNT(*), 6) AS fraud_rate
FROM paysim
GROUP BY type
ORDER BY fraud_rate DESC;
""").df()

type_risk

,type,total_transactions,fraud_transactions,fraud_rate
0,TRANSFER,532909,4097.0,0.007688
1,CASH_OUT,2237500,4116.0,0.001840
2,CASH_IN,1399284,0.0,0.000000
3,DEBIT,41432,0.0,0.000000
4,PAYMENT,2151495,0.0,0.000000


In [5]:
amount_buckets = con.execute("""
SELECT
    CASE
        WHEN amount < 50000 THEN '01: Under 50K'
        WHEN amount >= 50000 AND amount < 100000 THEN '02: 50K-100K'
        WHEN amount >= 100000 AND amount < 200000 THEN '03: 100K-200K'
        WHEN amount >= 200000 AND amount < 500000 THEN '04: 200K-500K'
        WHEN amount >= 500000 AND amount < 1000000 THEN '05: 500K-1M'
        ELSE '06: Over 1M'
    END AS amount_bucket,

    COUNT(*) AS total_transactions,
    SUM(isFraud) AS fraud_transactions,
    ROUND(SUM(isFraud) * 1.0 / COUNT(*), 6) AS fraud_rate

FROM paysim
GROUP BY amount_bucket
ORDER BY amount_bucket;
""").df()

amount_buckets

,amount_bucket,total_transactions,fraud_transactions,fraud_rate
0,01: Under 50K,2805947,1038.0,0.000370
1,02: 50K-100K,719309,669.0,0.000930
2,03: 100K-200K,1163794,1035.0,0.000889
3,04: 200K-500K,1333286,1607.0,0.001205
4,05: 500K-1M,209658,1158.0,0.005523
5,06: Over 1M,130626,2706.0,0.020716


In [6]:
depletion_signal = con.execute("""
SELECT
    isFraud,

    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1
            ELSE 0
        END
    ) AS high_depletion_transactions,

    ROUND(
        SUM(
            CASE
                WHEN oldbalanceOrg > 0
                     AND amount >= oldbalanceOrg * 0.90
                THEN 1
                ELSE 0
            END
        ) * 1.0 / COUNT(*),
        4
    ) AS high_depletion_rate

FROM paysim
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY isFraud
ORDER BY isFraud DESC;
""").df()

depletion_signal

,isFraud,total_transactions,high_depletion_transactions,high_depletion_rate
0,1,8213,8036.0,0.9784
1,0,2762196,1200755.0,0.4347


In [7]:
destination_change = con.execute("""
SELECT
    isFraud,
    COUNT(*) AS total_transactions,

    ROUND(AVG(amount), 2) AS avg_amount,

    ROUND(AVG(oldbalanceDest), 2) AS avg_oldbalance_dest,
    ROUND(AVG(newbalanceDest), 2) AS avg_newbalance_dest,

    ROUND(AVG(newbalanceDest - oldbalanceDest), 2) AS avg_dest_balance_change

FROM paysim
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY isFraud
ORDER BY isFraud DESC;
""").df()

destination_change

,isFraud,total_transactions,avg_amount,avg_oldbalance_dest,avg_newbalance_dest,avg_dest_balance_change
0,1,8213,1467967.3,544249.62,1279707.62,735458.00
1,0,2762196,314115.5,1706998.18,2052024.00,345025.82


In [8]:
destination_mismatch = con.execute("""
SELECT
    isFraud,
    COUNT(*) AS total_transactions,

    SUM(
        CASE
            WHEN amount > 0
                 AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
            THEN 1
            ELSE 0
        END
    ) AS destination_mismatch_transactions,

    ROUND(
        SUM(
            CASE
                WHEN amount > 0
                     AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
                THEN 1
                ELSE 0
            END
        ) * 1.0 / COUNT(*),
        4
    ) AS destination_mismatch_rate

FROM paysim
WHERE type IN ('TRANSFER', 'CASH_OUT')
GROUP BY isFraud
ORDER BY isFraud DESC;
""").df()

destination_mismatch

,isFraud,total_transactions,destination_mismatch_transactions,destination_mismatch_rate
0,1,8213,4236.0,0.5158
1,0,2762196,265485.0,0.0961


In [9]:
fraud_signal_summary = con.execute("""
WITH signal_flags AS (
    SELECT
        isFraud,

        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
            THEN 1 ELSE 0
        END AS risky_type_flag,

        CASE
            WHEN amount >= 500000
            THEN 1 ELSE 0
        END AS high_amount_flag,

        CASE
            WHEN oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1 ELSE 0
        END AS high_depletion_flag,

        CASE
            WHEN amount > 0
                 AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
            THEN 1 ELSE 0
        END AS destination_mismatch_flag

    FROM paysim
)

SELECT
    isFraud,
    COUNT(*) AS total_transactions,

    ROUND(AVG(risky_type_flag), 4) AS risky_type_rate,
    ROUND(AVG(high_amount_flag), 4) AS high_amount_rate,
    ROUND(AVG(high_depletion_flag), 4) AS high_depletion_rate,
    ROUND(AVG(destination_mismatch_flag), 4) AS destination_mismatch_rate

FROM signal_flags
GROUP BY isFraud
ORDER BY isFraud DESC;
""").df()

fraud_signal_summary

,isFraud,total_transactions,risky_type_rate,high_amount_rate,high_depletion_rate,destination_mismatch_rate
0,1,8213,1.0000,0.4705,0.9784,0.5158
1,0,6354407,0.4347,0.0529,0.3191,0.6011


In [10]:
fraud_signal_summary = con.execute("""
WITH signal_flags AS (
    SELECT
        isFraud,
        type,

        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
            THEN 1 ELSE 0
        END AS risky_type_flag,

        CASE
            WHEN amount >= 500000
            THEN 1 ELSE 0
        END AS high_amount_flag,

        CASE
            WHEN oldbalanceOrg > 0
                 AND amount >= oldbalanceOrg * 0.90
            THEN 1 ELSE 0
        END AS high_depletion_flag,

        CASE
            WHEN type IN ('TRANSFER', 'CASH_OUT')
                 AND amount > 0
                 AND ABS((newbalanceDest - oldbalanceDest) - amount) > 1
            THEN 1 ELSE 0
        END AS destination_mismatch_flag

    FROM paysim
)

SELECT
    isFraud,
    COUNT(*) AS total_transactions,

    ROUND(AVG(risky_type_flag), 4) AS risky_type_rate,
    ROUND(AVG(high_amount_flag), 4) AS high_amount_rate,
    ROUND(AVG(high_depletion_flag), 4) AS high_depletion_rate,

    ROUND(
        SUM(destination_mismatch_flag) * 1.0
        / NULLIF(SUM(risky_type_flag), 0),
        4
    ) AS destination_mismatch_rate_within_risky_types

FROM signal_flags
GROUP BY isFraud
ORDER BY isFraud DESC;
""").df()

fraud_signal_summary

,isFraud,total_transactions,risky_type_rate,high_amount_rate,high_depletion_rate,destination_mismatch_rate_within_risky_types
0,1,8213,1.0000,0.4705,0.9784,0.5158
1,0,6354407,0.4347,0.0529,0.3191,0.0961


In [11]:
fraud_signal_summary.to_csv("../outputs/fraud_signal_summary.csv", index=False)